# Fair PCA Comparison: PCA_MP vs fair_pca_multisource

This notebook summarizes the results from `simu_fairpca.py` comparing PCA_MP (method='fair') and fair_pca_multisource across different dimensions.

## Summary Table

The table shows:
- **MP (s)**: Average computation time for PCA_MP with standard deviation
- **SDP (s)**: Average computation time for fair_pca_multisource (SDP-based) with standard deviation
- **Speedup Ratio**: Ratio of SDP time / MP time (>1 means SDP is slower, <1 means MP is slower)


In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")


In [ ]:
# Load results
results_dir = "saved_results"
d_list = [30, 50, 70, 100, 200, 300]  # Dimensions to summarize

# Load data from individual dimension files or combined file
all_results = []

# Try loading combined file first
combined_file = os.path.join(results_dir, "results_fairpca_all.pkl")
if os.path.exists(combined_file):
    with open(combined_file, 'rb') as f:
        all_results = pickle.load(f)
    print(f"Loaded {len(all_results)} results from combined file")
else:
    # Load from individual files
    for d in d_list:
        fname = os.path.join(results_dir, f"results_fairpca_d{d}.pkl")
        if os.path.exists(fname):
            with open(fname, 'rb') as f:
                results_d = pickle.load(f)
                all_results.extend(results_d)
            print(f"Loaded {len(results_d)} results for d={d}")

print(f"Total results loaded: {len(all_results)}")

# Filter to only successful runs (no errors)
all_results = [r for r in all_results if 'error' not in r and 'd' in r]
print(f"Successful results: {len(all_results)}")


In [ ]:
# Compute statistics for each dimension
summary_data = []

for d in d_list:
    d_results = [r for r in all_results if r.get('d') == d]
    
    if len(d_results) == 0:
        print(f"Warning: No results found for d={d}")
        continue
    
    # Extract times and compute statistics
    times_mp = [r['time_mp'] for r in d_results if 'time_mp' in r]
    times_fair = [r['time_fair'] for r in d_results if 'time_fair' in r]
    
    if len(times_mp) > 0 and len(times_fair) > 0:
        avg_time_mp = np.mean(times_mp)
        std_time_mp = np.std(times_mp)
        avg_time_fair = np.mean(times_fair)
        std_time_fair = np.std(times_fair)
        
        # Speedup ratio (SDP time / MP time)
        speedup_ratios = [t_fair / t_mp for t_mp, t_fair in zip(times_mp, times_fair) if t_mp > 0]
        avg_speedup = np.mean(speedup_ratios) if len(speedup_ratios) > 0 else np.nan
        std_speedup = np.std(speedup_ratios) if len(speedup_ratios) > 0 else np.nan
        
        summary_data.append({
            'd': d,
            'MP_mean': avg_time_mp,
            'MP_std': std_time_mp,
            'SDP_mean': avg_time_fair,
            'SDP_std': std_time_fair,
            'Speedup_mean': avg_speedup,
            'Speedup_std': std_speedup,
            'n_runs': len(d_results)
        })
        
        print(f"d={d:3d}: MP={avg_time_mp:.3f}±{std_time_mp:.3f}s, "
              f"SDP={avg_time_fair:.3f}±{std_time_fair:.3f}s, "
              f"Speedup={avg_speedup:.2f}±{std_speedup:.2f}x, "
              f"n={len(d_results)}")
    else:
        print(f"Warning: Missing time data for d={d}")


In [ ]:
# Create summary table
if len(summary_data) > 0:
    # Create DataFrame for easier manipulation
    df = pd.DataFrame(summary_data)
    
    # Format data for display table
    table_rows = []
    
    # Row 1: MP times
    mp_row = {'Method': 'MP (s)'}
    for d in d_list:
        row_data = df[df['d'] == d]
        if len(row_data) > 0:
            mean_val = row_data.iloc[0]['MP_mean']
            std_val = row_data.iloc[0]['MP_std']
            if mean_val < 1:
                mp_row[f'd={d}'] = f"{mean_val:.3f}±{std_val:.3f}"
            elif mean_val < 60:
                mp_row[f'd={d}'] = f"{mean_val:.2f}±{std_val:.2f}"
            else:
                mp_row[f'd={d}'] = f"{mean_val:.1f}±{std_val:.1f}"
        else:
            mp_row[f'd={d}'] = "N/A"
    table_rows.append(mp_row)
    
    # Row 2: SDP times
    sdp_row = {'Method': 'SDP (s)'}
    for d in d_list:
        row_data = df[df['d'] == d]
        if len(row_data) > 0:
            mean_val = row_data.iloc[0]['SDP_mean']
            std_val = row_data.iloc[0]['SDP_std']
            if mean_val < 1:
                sdp_row[f'd={d}'] = f"{mean_val:.3f}±{std_val:.3f}"
            elif mean_val < 60:
                sdp_row[f'd={d}'] = f"{mean_val:.2f}±{std_val:.2f}"
            else:
                sdp_row[f'd={d}'] = f"{mean_val:.1f}±{std_val:.1f}"
        else:
            sdp_row[f'd={d}'] = "N/A"
    table_rows.append(sdp_row)
    
    # Row 3: Speedup ratio
    speedup_row = {'Method': 'Speedup Ratio'}
    for d in d_list:
        row_data = df[df['d'] == d]
        if len(row_data) > 0:
            mean_val = row_data.iloc[0]['Speedup_mean']
            std_val = row_data.iloc[0]['Speedup_std']
            if not np.isnan(mean_val):
                speedup_row[f'd={d}'] = f"{mean_val:.2f}±{std_val:.2f}"
            else:
                speedup_row[f'd={d}'] = "N/A"
        else:
            speedup_row[f'd={d}'] = "N/A"
    table_rows.append(speedup_row)
    
    # Create summary DataFrame
    summary_df = pd.DataFrame(table_rows)
    
    # Display table
    print("=" * 80)
    print("Summary Table: Computation Time Comparison")
    print("=" * 80)
    display(summary_df)
    
    # Also save as LaTeX table
    latex_str = summary_df.to_latex(index=False, float_format='%.2f')
    print("\nLaTeX table:")
    print(latex_str)
